# Environments

**A `requirements.txt` is fast to reason about because it is a declaration of names, not a search.** An environment manifest works the same way: parsing it takes milliseconds and yields the skeleton graph before a single file is read.

> **Every cell in this notebook runs.** They are generated from
> [`tools/notebooks/spec.py`](../tools/notebooks/spec.py) and executed by CI, so a
> cell that cannot run does not reach a commit. Change a cell, re-run it, and the
> page is yours — that is what it is for.


```
DECLARE  →  BIND  →  ATTACH  →  CORROBORATE  →  UNDERSTAND
manifest    sim/live  register    discovery      layers + answer
(ms)        one tag   capability  evidence
```

**One tag swaps everything.** `target: simulated | live` decides which connector
answers. Same manifest, same code paths, same answers — only the binding differs.
You prove a case in the simulator, then point the identical configuration at the
real thing.

And the delta between what was *declared* and what was *observed* is first-class
intelligence, not an error: declared-but-absent is a dead declaration, and
observed-but-undeclared is a shadow dependency.

In [ ]:
# --- setup: works locally, on Binder, and on Colab -------------------------
import subprocess, sys, pathlib

def _ensure_installed():
    """Install the package if it is not importable. No-op when it already is."""
    try:
        import slpie, gratimos          # noqa: F401
        return pathlib.Path(slpie.__file__).parent.parent
    except ModuleNotFoundError:
        pass
    here = pathlib.Path.cwd()
    root = next(
        (p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), None,
    )
    if root is None:                     # Colab: no checkout, so fetch one
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Reimain/Macropol-s.git", "/content/Macropol-s"],
            check=True,
        )
        root = pathlib.Path("/content/Macropol-s")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)],
                   check=True)
    sys.path.insert(0, str(root))
    return root

ROOT = _ensure_installed()
print("package root:", ROOT)

import slpie
print("slpie", slpie.__version__)

## Write a manifest

In [ ]:
import pathlib, tempfile
from slpie.environment import loads, scaffold

WORK = pathlib.Path(tempfile.mkdtemp(prefix="slpie-nb-"))

MANIFEST = """
apiVersion: slpie/v1
environment: acme-production
target: simulated

security:
  concerns: [pci-dss, gdpr]
  boundaries:
    - name: cardholder-data
      contains: [payments]

codebase:
  - root: ./services/payments
    language: npm
    team: payments
  - root: ./services/orders
    language: python
    team: fulfilment

network:
  - name: payments-api
    url: https://api.acme.com/v1
    kind: rest

data:
  - folder: ./warehouse/orders
    kind: schema

providers:
  - name: stripe
"""

path = WORK / "slpie.environment.yaml"
path.write_text(MANIFEST)

manifest = loads(MANIFEST, source_uri=path.resolve().as_uri())
print("environment:", manifest.environment)
print("target:     ", manifest.target.value)
print("declared:   ", len(manifest.declarations), "element(s)")
print()
for declaration in manifest:
    print(f"  {declaration.kind.value:12} {declaration.name:12} {declaration.location}")

## The skeleton graph exists before anything is read

A declaration is evidence — `DECLARED`, at 0.92. Authoritative about *intent*, and deliberately not about reality.

In [ ]:
from slpie.engine import Engine

engine = Engine.from_manifest(str(path))
count = engine.declare()

print(f"{count} node(s) in the graph, with no file read yet")
print("graph counts:", engine.graph.counts())

## Materialise the declared world

The simulator writes **real artifacts**, not mocks.

In [ ]:
world = engine.simulate(root=str(WORK / "world"))

print("world at:", world.root)
print()
for artifact in sorted(world.artifacts, key=lambda item: str(item.path))[:12]:
    print(f"  {artifact.kind:14} {artifact.path.relative_to(world.root)}")

Real `package-lock.json`, real `go.mod`, real Kubernetes YAML, a real `git` repository with a real commit. The *same* discoverers run over this as over a customer's tree — so a green simulator case is evidence about the real code path, not about a mock.

In [ ]:
lock = world.read("payments", "package-lock.json")
print(lock[:320])

## Attach, and negotiate capabilities

In [ ]:
registrations = engine.attach()

for registration in registrations:
    granted = [c.name for c in registration.negotiation.granted]
    refused = [c.name for c in registration.negotiation.refused]
    print(f"  {registration.element:12} granted: {', '.join(granted[:3])}")
    if refused:
        print(f"  {'':12} refused: {', '.join(refused)}")

**A refused capability becomes a named gap on every answer whose confidence it limits.** That is what separates a low-confidence answer from a misleading one.

In [ ]:
report = engine.scan()
print("scan:", report)
print()
print("gaps the platform is carrying:")
for gap in engine.gaps()[:6]:
    print(f"  · {gap.kind.value:22} {gap.subject[:22]:24} {gap.detail[:34]}")

## Declared vs observed

In [ ]:
reconciliation = engine.reconcile()
print(reconciliation.summary())
print()
print("declared but never observed:", len(reconciliation.declared_not_found))
print("observed but never declared:", len(reconciliation.undeclared))
print("corroborated:               ", len(reconciliation.corroborated))

## Fire a condition at it

Twelve scenarios ship. Each **rewrites the world** and then expects the platform to *discover* the change — nothing is written to the graph directly, so the scenario tests the real path.

In [ ]:
from slpie.simulator.scenarios import available

print(f"{len(available())} scenarios:")
for name in available():
    print("  ", name)

In [ ]:
outcome = engine.fire("cve", package="lodash", version="4.17.20")

print("scenario:  ", outcome.scenario)
print("changed:   ", outcome.changed)
print("expects:   ", outcome.expect_findings or outcome.expect_gaps)
print("detail:    ", outcome.detail)
print()
print("The expectation is data on the outcome — so a test asserts it")
print("rather than a human reading the narration and nodding.")

In [ ]:
# Rescan, and see whether the platform found what the scenario planted.
engine.scan()
findings = engine.reconciliation_findings()
print(f"{len(findings)} finding(s) after the scenario")

engine.close()

## Your turn

Change `target: simulated` to `target: live` in the manifest above. The binding refuses without an explicit confirmation — that is the one dangerous action in the platform, and it is gated in exactly one place (`slpie/binding/guard.py`) rather than reimplemented per surface.

In [ ]:
# Scratch cell — fire a different scenario.
engine2 = Engine.from_manifest(str(path))
engine2.declare()
engine2.simulate(root=str(WORK / "world2"))
engine2.attach()

for name in ("boundary-breach", "shadow-dependency", "license-change"):
    result = engine2.fire(name)
    print(f"  {name:20} changed={str(result.changed):5} expects={result.expect_findings}")

engine2.close()